# LoRA vs prompting: the GPU runs

This notebook runs the GPU half of [lora-vs-prompting](https://github.com/jpgg2105/lora-vs-prompting) on a free Colab T4:

1. Qwen2.5-1.5B-Instruct **zero-shot** and **few-shot** (10 examples retrieved from a pool of 100, 500 or 2,000 labelled examples)
2. The same model **fine-tuned with LoRA** on each pool (three seeds for the pool of 100)
3. Predictions for the same 300 test utterances, with GPU time and token counts

The Gemini few-shot baseline runs separately on your computer (`scripts\run_gemini.ps1`), and the report combines both.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. The full run takes about an hour. Results are saved to Google Drive as they finish, so if Colab disconnects, run all cells again and it picks up where it stopped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
import os, pathlib
REPO = "https://github.com/jpgg2105/lora-vs-prompting.git"
if not pathlib.Path("/content/lora-vs-prompting").exists():
    if os.system(f"git clone -q {REPO} /content/lora-vs-prompting") != 0:
        from google.colab import files
        print("Could not clone the repo. Upload lora-vs-prompting.zip (the project folder, zipped) instead:")
        up = files.upload()
        os.system(f"unzip -q -o '{next(iter(up))}' -d /content")
%cd /content/lora-vs-prompting

In [ ]:
!pip install -q -e ".[gpu]"

## 2. Keep results on Google Drive
So nothing is lost if the session ends. Set `USE_DRIVE = False` to keep them in the session only.

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["LVP_RESULTS"] = "/content/drive/MyDrive/lora-vs-prompting/results"
print("Results go to", os.environ.get("LVP_RESULTS", "/content/lora-vs-prompting/results"))

## 3. Data
Downloads MASSIVE 1.1 (CC BY 4.0) and builds the same fixed splits as on your computer: pools of 100, 500 and 2,000 training examples and 300 test utterances (seed 13).

In [ ]:
!python -m lvp.cli prepare

## 4. Quick check (2 to 3 minutes)
20 test utterances and one short LoRA run, saved apart from the real results. If this works, the full run will too.

In [ ]:
!python -m lvp.cli gpu --quick

## 5. Full run (about an hour)
Safe to run again: finished runs are skipped.

In [ ]:
!python -m lvp.cli gpu

## 6. Report and download
The report here covers the Qwen runs. Download the zip and attach it to the chat; the Gemini files from your computer go in the same `results/preds` folder for the combined report.

In [ ]:
!python -m lvp.cli report
res = os.environ.get("LVP_RESULTS", "/content/lora-vs-prompting/results")
print(open(f"{res}/REPORT.md").read())
import shutil
shutil.make_archive("/content/lora-results", "zip", res)
from google.colab import files
files.download("/content/lora-results.zip")